In [1]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install sqlalchemy
!pip install scikit-learn
!pip install python-dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psycopg2
import os
from sqlalchemy import create_engine, URL


In [2]:
# Get connection to DB



from pathlib import Path
from dotenv import load_dotenv

#Get env varibles from FetchData folder
env_path = Path("../../Fetch_Data/.env").resolve()
load_dotenv(env_path)

# Verify correct Path
print(env_path)
print(env_path.exists())

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5433")),
    database=os.getenv("POSTGRES_DB")
)

engine = create_engine(url)

test_query = """
SELECT pokemon_name
FROM pokemon
"""

df = pd.read_sql(test_query, engine)
print(df.head())

C:\Users\brice\OneDrive\Documents\Projects\Pokemon_VGC\Pokemon_VGC_Analysis\Fetch_Data\.env
True
  pokemon_name
0    bulbasaur
1      ivysaur
2     venusaur
3   charmander
4   charmeleon


# Phase 1:

- Use logistical Regression to predict whether a team will make top cup (In this case defined as **top 10%**)
- First step consider a sub select of pokemon using one-hot encoding
    - Decide which pokemon to use via usage rates (only consider the most popular pokemon)
- Make different predictors per each regulation

## Example Encoding

| pk_1  | pk_2     | top_10%  |
|-------|----------|----------|
| 1     | 0        | 1        |
| 1     | 1        |0         |



## Modifable Options

- Regulations:
    - Champions: M-A. M-B, M-C (Check for Updates)
    - Scarlet/Violet: A, B, C, ... H, J
    - Sword/Sheild: S1, S2, S3, ... S13, S14

- Number of Pokemon: 
    - Alters the amount of Top pokemon that are retrieved, i.e. more pokemon = more data

- Min Event Size:
    - Alters the minimum event size to avoid skewed results giving inflated usage rates (Default set 60)



In [ ]:
from sklearn.linear_model import LogisticRegression
from sqlalchemy import text


sql_path = Path("top_pokemon_usage.sql")

with open(sql_path, "r") as file:
    query = text(file.read())
    
regulation = "I"
num_pokemon = 10
min_event_size = 60




top_n_usage = pd.read_sql(
    query,
    engine,
    params={
        "regulation" : regulation,
        "num_pokemon" : num_pokemon,
        "min_event_size" : min_event_size
    }
)

top_names = top_n_usage["pokemon_name"].tolist()
print(top_names)


team_query = text("""
    SELECT
        t.team_id,
        p.pokemon_name
    FROM teams t
    JOIN events e
        ON t.event_id = e.event_id
    JOIN team_pokemon tp
        ON t.team_id = tp.team_id
    JOIN pokemon p
        ON tp.pk_id = p.pk_id
    WHERE e.regulation = :regulation;
""")


team_pokemon = pd.read_sql(
    team_query,
    engine,
    params={
        "regulation" : regulation
    }
)

# Get rid of non-top_n pokemon, because that data is not helpful this model
filtered = team_pokemon[team_pokemon["pokemon_name"].isin(top_names)]



